# Does Your Neighbourhood Determine Your Business's Fate?
## A Survival Analysis of 97,915 Vancouver Businesses (2013–2026)

A friend on Main Street asked me if his ice cream shop would survive the Broadway Subway construction. I didn't have an answer. So I pulled 13 years of Vancouver business licence data, joined it to property tax assessments, and ran the numbers. Here's what the data says about where he stands.

In [ ]:
from IPython.display import Image, display
display(Image(filename='plots/tldr_1_survival_curve.png'))

In [ ]:
display(Image(filename='plots/tldr_2_type_effect.png'))

In [ ]:
display(Image(filename='plots/tldr_3_neighbourhood_effect.png'))

### The short version

**The typical Vancouver business lasts 6 years.** Not the year-one cliff that folklore suggests — a steady drain through year five.

**What kind of business you run matters most.** Food & Beverage has nearly 2× the closure risk of the baseline. Construction & Trades is the most durable. Business type dwarfs neighbourhood effects.

**Mount Pleasant is not the problem.** Once you control for its heavier mix of restaurants and retail, it's statistically indistinguishable from Downtown.

**Land value is a weak signal.** Higher assessed values add ~7% closure risk per doubling, but the variable is a residential property proxy — suggestive, not definitive.

*Full methodology and data pipeline on [GitHub](https://github.com/7AVS/portfolio).*

---

## Step 1: How Long Does a Vancouver Business Actually Last?

Before asking whether neighbourhood matters, I need a baseline. I used a Kaplan-Meier estimator — the same survival analysis method hospitals use to track patient outcomes, applied here to 97,915 independent businesses. Each time a business closes, the curve drops. Businesses still operating at the end aren't counted as failures — they're unknowns (statisticians call this "right-censoring," and the KM method handles it correctly). The dashed line on the chart is a restricted cohort (post-2013 entrants only) as a sensitivity check I'll explain below.

*A note on the data: I reclassified 9,215 businesses that stopped renewing licences without a formal closure — status still showing "Issued" but no activity for 3+ years. That correction alone drops the median from 8 to 6 years. I also filtered to independent businesses only (chains skew the picture). About 40% of the sample was already operating when the data starts in 2013, which means I don't know their true founding date — a bias called "left-truncation" that makes long-lived businesses look more common. I run a restricted cohort (2014+ entrants only) throughout as a check. The results hold.*

In [ ]:
from IPython.display import Image, display
display(Image(filename='plots/step1_baseline_survival.png'))

### Key Statistics

| Milestone | Survival Rate | 95% CI |
|-----------|--------------|--------|
| 1 year | 88.8% | [88.6, 89.0] |
| 3 years | 68.5% | [68.2, 68.8] |
| 5 years | 52.5% | [52.2, 52.8] |
| 7 years | 43.1% | [42.8, 43.5] |
| 10 years | 34.1% | [33.7, 34.4] |

Median survival: **6.0 years** (95% CI: 5–6).

*Independent businesses only (97,915). Includes 9,215 silent exits reclassified as closures.*

The "most businesses fail in year one" story is wrong. It's a steady drain through year five, not a cliff. The curve flattens after year seven — the fragile exits have cleared out, and what's left skews durable. For a business opening today, **6 years is the honest number**.

### Has it always been this way?

Six years is the median — but is it getting better or worse? The chart below tracks 1-year, 2-year, and 3-year survival rates for each entry-year cohort from 2013 to 2024.

In [ ]:
import warnings
warnings.filterwarnings("ignore", module="lifelines")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from lifelines import KaplanMeierFitter
from IPython.display import Image, display

PROJECT = "/home/aurora/projects/sites/portfolio-projects/van-property-tax"

# ── Load panel ────────────────────────────────────────────────────────────────
df = pd.read_csv(f"{PROJECT}/data/processed/commercial-panel-v3.csv", low_memory=False)
df = df[df["is_chain"] == False].copy()
df["duration"] = df["years_active"]
df["event"]    = df["event_v3"].astype(int)
df["entry_year_4d"] = df["first_year"].apply(lambda y: 2000 + int(y) if int(y) < 100 else int(y))

# ── Compute 1yr / 2yr / 3yr KM survival per entry cohort ─────────────────────
def km_at(kmf, t):
    """Return KM survival probability at time t, or None if t not yet reached."""
    tl = kmf.timeline
    sf = kmf.survival_function_
    idx = tl[tl <= t]
    if len(idx) == 0:
        return None
    return float(sf.loc[idx[-1]].iloc[0])

rows = []
for yr in range(2013, 2025):
    cohort = df[df["entry_year_4d"] == yr]
    if len(cohort) < 50:
        continue
    kmf = KaplanMeierFitter()
    kmf.fit(cohort["duration"], event_observed=cohort["event"])
    rows.append({
        "year": yr,
        "n":    len(cohort),
        "s1":   km_at(kmf, 1),
        "s2":   km_at(kmf, 2),
        "s3":   km_at(kmf, 3),
    })

cohort_df = pd.DataFrame(rows)

# ── Plot ──────────────────────────────────────────────────────────────────────
GREEN  = "#2eaa5e"
BLUE   = "#4472c4"
ORANGE = "#e07b39"

YEARS  = cohort_df["year"].values
S1     = cohort_df["s1"].values * 100
S2     = cohort_df["s2"].values * 100
S3     = cohort_df["s3"].values * 100

fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

ax.plot(YEARS, S1, color=GREEN,  linewidth=2.0, marker="o", markersize=6, label="1-year survival")
ax.plot(YEARS, S2, color=BLUE,   linewidth=2.0, marker="s", markersize=6, label="2-year survival")
ax.plot(YEARS, S3, color=ORANGE, linewidth=2.0, marker="D", markersize=5, label="3-year survival",
        zorder=4)

ax.grid(axis="y", color="#eeeeee", linewidth=0.7, zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#dddddd")
ax.spines["bottom"].set_color("#dddddd")
ax.tick_params(axis="both", labelsize=9, color="#dddddd")
ax.set_xlim(2012.4, 2024.8)
ax.set_ylim(48, 100)
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
ax.set_xlabel("Entry Year", fontsize=11, labelpad=8)
ax.set_ylabel("Survival Rate (%)", fontsize=11, labelpad=8)
ax.set_title(
    "Early Survival by Entry Year Cohort — Are Newer Businesses Lasting Longer?",
    fontsize=13, fontweight="bold", pad=14,
)

# ── Selective annotations — first/last/min/max per series ─────────────────────
def select_years(vals, years):
    idx_min = int(np.argmin(vals))
    idx_max = int(np.argmax(vals))
    return {years[0], years[-1], years[idx_min], years[idx_max]}

chosen_s1 = select_years(S1, YEARS)
chosen_s2 = select_years(S2, YEARS)
chosen_s3 = select_years(S3, YEARS)

def annotate(ax, x, y, label, color, y_off=0):
    ax.annotate(label, xy=(x, y), xytext=(x + 0.08, y + 1.0 + y_off),
                fontsize=8.0, color=color, fontweight="semibold")

for i, yr in enumerate(YEARS):
    if yr in chosen_s1:
        annotate(ax, yr, S1[i], f"{S1[i]:.1f}%", GREEN)
    if yr in chosen_s2:
        annotate(ax, yr, S2[i], f"{S2[i]:.1f}%", BLUE)
    if yr in chosen_s3:
        annotate(ax, yr, S3[i], f"{S3[i]:.1f}%", ORANGE)

ax.legend(loc="lower right", fontsize=9, frameon=True, framealpha=0.92, edgecolor="#dddddd")

fig.text(0.01, 0.01,
         "Source: City of Vancouver Open Data. Independent businesses only. "
         "event_v3 corrected. Recent cohorts have shorter observation windows.",
         fontsize=7, color="#888888")

plt.tight_layout(rect=[0, 0.04, 1, 1])
PLOT_PATH = f"{PROJECT}/analysis/plots/step1b_cohort_early_survival.png"
plt.savefig(PLOT_PATH, dpi=150, bbox_inches="tight", facecolor="white", edgecolor="none")
plt.close()

display(Image(filename="plots/step1b_cohort_early_survival.png"))

Early survival has been improving. 2015 was the trough — only 79.5% of businesses survived year one. By 2023, it's 93%. The 2020 cohort shows a spike that likely reflects COVID-era effects — fewer businesses opened, and those that did may have been more deliberate.

Two caveats: recent cohorts have shorter observation windows (the 2024 three-year rate will change as more data comes in), and the 2013 cohort includes businesses already operating when the data starts.

The aggregate curve treats a yoga studio and an auto repair shop as the same thing. They're not — the gap between Financial & Insurance and Tech is 6 years of median survival.

## Step 2: Does the Type of Business Matter?

I split the 97,915 independent businesses by their macro-category and ran separate survival curves for the top 9 types. I also grouped them by a "gentrification signal" — whether their industry is typically associated with neighbourhood change. Tech firms, creative studios, and fitness centres get a HIGH signal; stable trades and essential services get LOW.

In [ ]:
import warnings
warnings.filterwarnings("ignore", module="lifelines")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from lifelines import KaplanMeierFitter

# ── Load & join ─────────────────────────────────────────────────────────────
PROJECT = "/home/aurora/projects/sites/portfolio-projects/van-property-tax"

df = pd.read_csv(f"{PROJECT}/data/processed/commercial-panel-v3.csv", low_memory=False)
macro = pd.read_csv(f"{PROJECT}/data/processed/businesstype_macro_categories.csv")

# Strip *Historic* suffix before joining
df["_bt"] = df["businesstype"].str.replace(r"\s*\*Historic\*$", "", regex=True).str.strip()
macro["_bt"] = macro["businesstype"].str.replace(r"\s*\*Historic\*$", "", regex=True).str.strip()
macro_dedup = macro[["_bt", "macro_category", "gentrification_signal"]].drop_duplicates(subset=["_bt"])
df = df.merge(macro_dedup, on="_bt", how="left").drop(columns=["_bt"])

# ── Filter to independent businesses only ────────────────────────────────────
df = df[df["is_chain"] == False].copy()

# ── Survival variables (event_v3 corrected) ──────────────────────────────────
df["duration"] = df["years_active"]
df["event"]    = df["event_v3"].astype(int)

# ── Fit KM by top 9 macro-categories ────────────────────────────────────────
TOP_N    = 9
top_cats = df["macro_category"].value_counts().head(TOP_N).index.tolist()

# Distinct palette — avoid similar hues
TYPE_COLORS = [
    "#1a4a6b", "#e07b39", "#2a9d5c", "#9b59b6", "#d4a017",
    "#c0392b", "#16a085", "#555555", "#2980b9",
]

fig, ax = plt.subplots(figsize=(11, 7))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

for i, cat in enumerate(top_cats):
    sub = df[df["macro_category"] == cat]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["duration"], event_observed=sub["event"])
    t  = kmf.timeline
    sf = kmf.survival_function_.values.flatten()
    ax.step(t, sf, where="post", color=TYPE_COLORS[i], linewidth=1.8,
            label=f"{cat} (n={len(sub):,})")

ax.axhline(y=0.5, color="#999999", linewidth=0.8, linestyle="--", alpha=0.6)

ax.set_xlim(-0.2, 14.5)
ax.set_ylim(-0.02, 1.05)
ax.set_xlabel("Years in business", fontsize=11, labelpad=8)
ax.set_ylabel("Probability of survival", fontsize=11, labelpad=8)
ax.set_title("Business Survival by Type — Independent Businesses (2013–2025)",
             fontsize=13, fontweight="bold", pad=14)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#dddddd")
ax.spines["bottom"].set_color("#dddddd")
ax.tick_params(axis="both", labelsize=9, color="#dddddd")
ax.grid(axis="y", color="#eeeeee", linewidth=0.7)

# Legend OUTSIDE the plot area — below, 2 columns, so it doesn't overlap curves
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=8,
          frameon=True, framealpha=0.92, edgecolor="#dddddd", columnspacing=1.5)

fig.text(0.01, 0.01,
         "Source: City of Vancouver Open Data — Business Licences 2013–2026. "
         "Independent businesses only. event_v3 corrected.",
         fontsize=7, color="#888888")
plt.tight_layout(rect=[0, 0.10, 1, 1])
plt.savefig(f"{PROJECT}/analysis/plots/step2_survival_by_type.png", dpi=150,
            bbox_inches="tight", facecolor="white", edgecolor="none")
plt.close()

from IPython.display import Image, display
display(Image(filename="plots/step2_survival_by_type.png"))

Type matters a lot. Financial & Insurance has an 11-year median and 68% at five years — best in the panel. Health & Beauty surprised me at 8 years. Technology & Information is the worst at 5 years, only 44% surviving to year five. The spread between the most and least durable categories is 6 years.

### Survival by Business Type — Top 9 Categories

| Category | Count | Median Survival | 5-yr Rate |
|---|---|---|---|
| Construction & Trades | 20,936 | 6 yr | 50.8% |
| Professional Services | 19,290 | 5 yr | 49.3% |
| Health & Beauty | 16,292 | 8 yr | 61.1% |
| Food & Beverage | 10,625 | 6 yr | 51.9% |
| Retail | 7,417 | 5 yr | 48.8% |
| Arts & Culture | 3,567 | 7 yr | 53.9% |
| Manufacturing & Wholesale | 3,093 | 6 yr | 53.6% |
| Financial & Insurance | 3,056 | 11 yr | 67.9% |
| Technology & Information | 3,039 | 5 yr | 44.4% |

*Independent businesses only. event_v3 corrected — includes silent exit reclassification.*

### Does the gentrification narrative hold up?

Type matters — but do the *kinds* of businesses associated with neighbourhood change actually die faster?

In [ ]:
# ── Fit KM by gentrification signal ─────────────────────────────────────────
signal_order  = ["HIGH", "MEDIUM", "LOW", "NEUTRAL"]
signal_colors = {
    "HIGH":    "#e63946",
    "MEDIUM":  "#f4a261",
    "LOW":     "#457b9d",
    "NEUTRAL": "#aaaaaa",
}

df["signal_upper"] = df["gentrification_signal"].str.upper()

fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

for sig in signal_order:
    sub = df[df["signal_upper"] == sig]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["duration"], event_observed=sub["event"])
    t  = kmf.timeline
    sf = kmf.survival_function_.values.flatten()
    ci = kmf.confidence_interval_survival_function_
    color = signal_colors[sig]
    ax.step(t, sf, where="post", color=color, linewidth=2.2,
            label=f"{sig} signal (n={len(sub):,})", zorder=5)
    ax.fill_between(t, ci.iloc[:, 0].values, ci.iloc[:, 1].values,
                    step="post", color=color, alpha=0.08, zorder=3)

ax.axhline(y=0.5, color="#999999", linewidth=0.8, linestyle="--", alpha=0.6)

ax.set_xlim(-0.2, 14.5)
ax.set_ylim(-0.02, 1.05)
ax.set_xlabel("Years in business", fontsize=11, labelpad=8)
ax.set_ylabel("Probability of survival", fontsize=11, labelpad=8)
ax.set_title("Business Survival by Gentrification Signal — Independent Businesses (2013–2025)",
             fontsize=12, fontweight="bold", pad=14)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#dddddd")
ax.spines["bottom"].set_color("#dddddd")
ax.tick_params(axis="both", labelsize=9, color="#dddddd")
ax.grid(axis="y", color="#eeeeee", linewidth=0.7)

# Legend below plot to avoid overlapping the HIGH curve
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=4, fontsize=9,
          frameon=True, framealpha=0.92, edgecolor="#dddddd")

fig.text(0.01, 0.01,
         "Source: City of Vancouver Open Data — Business Licences 2013–2026. "
         "Independent businesses only. event_v3 corrected.",
         fontsize=7, color="#888888")
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(f"{PROJECT}/analysis/plots/step2_gentrification_signal.png", dpi=150,
            bbox_inches="tight", facecolor="white", edgecolor="none")
plt.close()

from IPython.display import Image, display
display(Image(filename="plots/step2_gentrification_signal.png"))

No. HIGH-signal businesses — production companies, design studios, fitness centres — have the *best* survival in the panel at 13 years. That seems contradictory since Tech has a 5-year median, but the HIGH grouping includes durable niche subcategories (independent film, architecture, artist studios) that the broader Tech category doesn't capture. The signal classification is too coarse to do predictive work on its own. I'll drop it from the regression in Step 5.

Food & Beverage — my friend's category — sits at 6 years. The question is whether the block he's on makes that better or worse.

## Step 3: Does the Block Matter?

A Food & Beverage business on Main Street and one in Kitsilano are in very different environments — rents, foot traffic, transit, competition. My friend's ice cream shop is in Mount Pleasant. I split the 64,790 businesses with a mapped neighbourhood across Vancouver's 22 local areas.

In [ ]:
from IPython.display import Image, display
display(Image(filename='plots/step3_survival_by_neighbourhood.png'))

### Top 10 Neighbourhoods by Business Count

| Neighbourhood | Count | Median Survival | 5-yr Rate |
|---|---|---|---|
| Downtown | 21,595 | 8 yr | 61.1% |
| Fairview | 6,607 | 10 yr | 67.6% |
| Mount Pleasant | 4,708 | 7 yr | 58.4% |
| West End | 3,580 | 8 yr | 61.7% |
| Kitsilano | 3,571 | 10 yr | 67.2% |
| Grandview-Woodland | 2,697 | 9 yr | 61.5% |
| Kensington-Cedar Cottage | 2,670 | 13 yr | 70.7% |
| Strathcona | 2,447 | 8 yr | 61.2% |
| Sunset | 2,366 | 10 yr | 65.7% |
| Renfrew-Collingwood | 2,246 | 13 yr | 70.6% |

*Independent businesses only (64,790 with mapped neighbourhood). event_v3 corrected.*

In [ ]:
import warnings
warnings.filterwarnings("ignore", module="lifelines")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from lifelines import KaplanMeierFitter

PROJECT = "/home/aurora/projects/sites/portfolio-projects/van-property-tax"

df_nb = pd.read_csv(f"{PROJECT}/data/processed/commercial-panel-v3.csv", low_memory=False)
df_nb = df_nb[df_nb["is_chain"] == False].copy()
df_nb = df_nb[df_nb["localarea"].notna() & (df_nb["localarea"] != "Out of Town")].copy()
df_nb["duration"] = df_nb["years_active"]
df_nb["event"]    = df_nb["event_v3"].astype(int)

# ── Fit KM for all neighbourhoods with n≥200 to rank best vs worst ──────────
ALL_THRESHOLD = 200
area_counts   = df_nb["localarea"].value_counts()
eligible      = area_counts[area_counts >= ALL_THRESHOLD].index.tolist()

all_stats = []
for nb in eligible:
    sub = df_nb[df_nb["localarea"] == nb]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["duration"], sub["event"], label=nb)
    med      = kmf.median_survival_time_
    med_rank = med if (not np.isinf(med) and not np.isnan(med)) else 15.0
    sf5      = kmf.survival_function_at_times([5]).values[0]
    all_stats.append({"neighbourhood": nb, "n": len(sub),
                      "median": med, "median_rank": med_rank,
                      "sf5": sf5, "kmf": kmf})

all_stats_df = pd.DataFrame(all_stats).sort_values("median_rank", ascending=False)
best3  = all_stats_df.head(3)["neighbourhood"].tolist()
worst3 = all_stats_df.tail(3)["neighbourhood"].tolist()
all_kmf_fits = {row["neighbourhood"]: {"kmf": row["kmf"], "n": row["n"]}
                for _, row in all_stats_df.iterrows()}

# ── Palette ─────────────────────────────────────────────────────────────────
TB_PALETTE = {
    best3[0]: "#1a7a4a",
    best3[1]: "#2ecc71",
    best3[2]: "#82e0aa",
    worst3[0]: "#922b21",
    worst3[1]: "#e74c3c",
    worst3[2]: "#f1948a",
}

# ── Plot ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

for group_label, group_names, dash in [("Best", best3, "solid"), ("Worst", worst3, "dashed")]:
    for nb in group_names:
        kmf   = all_kmf_fits[nb]["kmf"]
        n     = all_kmf_fits[nb]["n"]
        color = TB_PALETTE[nb]
        t     = kmf.timeline
        sf    = kmf.survival_function_.values.flatten()
        ci    = kmf.confidence_interval_survival_function_
        med   = kmf.median_survival_time_
        med_str = f"{med:.0f}yr" if (not np.isinf(med) and not np.isnan(med)) else ">14yr"
        label = f"{nb} (n={n:,}, median={med_str})"
        ax.step(t, sf, where="post", color=color, linewidth=2.2,
                linestyle=dash, label=label, zorder=5)
        ax.fill_between(t, ci.iloc[:, 0].values, ci.iloc[:, 1].values,
                        step="post", color=color, alpha=0.08, zorder=3)

ax.axhline(y=0.5, color="#cc3333", linewidth=1.0, linestyle="--", alpha=0.7, zorder=2)
ax.annotate("50% survival", xy=(0.02, 0.505), xycoords=("axes fraction", "data"),
            fontsize=8, color="#cc3333", va="bottom")

ax.annotate("— Solid: best-surviving", xy=(0.98, 0.98), xycoords="axes fraction",
            fontsize=8, color="#1a7a4a", ha="right", va="top", fontweight="bold")
ax.annotate("-- Dashed: worst-surviving", xy=(0.98, 0.94), xycoords="axes fraction",
            fontsize=8, color="#922b21", ha="right", va="top", fontweight="bold")

ax.set_xlim(left=0)
ax.set_ylim(0, 1.05)
ax.set_xlabel("Years Since First Licence", fontsize=11, color="#2c2c2c")
ax.set_ylabel("Survival Probability", fontsize=11, color="#2c2c2c")
ax.set_title("Business Survival — Best vs Worst Neighbourhoods in Vancouver (2013–2025)",
             fontsize=12, fontweight="bold", color="#2c2c2c", pad=12)
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
ax.xaxis.set_minor_locator(mticker.MultipleLocator(1))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#dddddd")
ax.spines["bottom"].set_color("#dddddd")
ax.tick_params(axis="both", which="major", labelsize=9, color="#dddddd")
ax.grid(axis="y", color="#eeeeee", linewidth=0.7, zorder=1)

# Legend below the chart — avoids overlap with survival curves
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=8.5,
          frameon=True, framealpha=0.92, edgecolor="#dddddd", columnspacing=1.2)

fig.text(0.01, 0.01,
         "Source: City of Vancouver Open Data — Business Licences 2013–2026. "
         "Independent businesses only. event_v3 corrected.",
         fontsize=7, color="#888888", ha="left", va="bottom")

plt.tight_layout()
plt.subplots_adjust(bottom=0.2)
plt.savefig(f"{PROJECT}/analysis/plots/step3_survival_top_vs_bottom.png",
            dpi=150, bbox_inches="tight", facecolor="white", edgecolor="none")
plt.close()

from IPython.display import Image, display
display(Image(filename="plots/step3_survival_top_vs_bottom.png"))

The spread is real. Kensington-Cedar Cottage and Renfrew-Collingwood — east-side, working-class neighbourhoods — are the standouts at 13 years median and 70%+ at five years. Mount Pleasant sits at 7 years, worse than the citywide baseline.

It's not as simple as "rich neighbourhood = durable business." K-CC and R-C aren't high-income areas — they have stable, lower-turnover business mixes. But this is confounded. Mount Pleasant has more restaurants and retail, which close faster everywhere. That 7-year median might look different once you control for business type. To pull those variables apart, I need a model — and the property tax data adds a third dimension.

## Step 4: The Land Underneath

For each business, I pulled the median assessed land value in its neighbourhood during the year it opened. This covers 53,805 businesses across 16 of Vancouver's 22 local areas.

**Important caveat:** these are assessed values across *all* properties — mostly residential. Downtown's median ($382K) comes in lower than Mount Pleasant ($441K) because it's dominated by condo strata lots. This captures relative price environments, not what businesses actually pay in rent.

In [ ]:
from IPython.display import Image, display
display(Image(filename='plots/step4_survival_by_land_value_tier.png'))

### Survival by Land Value Tier

Businesses split into three equal groups by the median assessed value in their neighbourhood when they opened.

| Tier | Count | Median Survival | 5-yr Rate |
|------|-------|----------------|-----------|
| LOW | 17,962 | 8 yr | 60.3% |
| MEDIUM | 17,984 | 9 yr | 64.0% |
| HIGH | 17,859 | 10 yr | 66.6% |

*Land values are neighbourhood-level residential-dominated assessments.*

In [ ]:
display(Image(filename='plots/step4_land_value_vs_survival_scatter.png'))

Not what the gentrification-pressure story predicts. Businesses in high-value areas survive *longer*, not shorter. But the neighbourhood-level relationship is noisy (Pearson r = 0.32, p = 0.23) — Killarney ($522K, 14-year median) sits well above the trend; Riley Park ($1.14M, 8 years) sits well below.

Mount Pleasant lands below the trendline — partly because it has more Food & Beverage, which closes faster everywhere. Land value alone doesn't explain the gap. To ask what *independently* predicts survival, I need a model that includes type, neighbourhood, and land value together.

## Step 5: Putting It All Together

Steps 2 through 4 each showed a factor linked to survival — but looked at each one alone. The problem: Mount Pleasant has more restaurants, and restaurants close faster everywhere. Is that a neighbourhood effect or a composition effect?

A Cox proportional hazards model (Cox PH) answers this — it's a regression that estimates the independent contribution of each variable while holding the others constant. I fit three nested models on 53,805 businesses with complete data. The output is a **hazard ratio** (HR) for each variable: HR > 1 means higher closure risk, HR < 1 means more durable, HR = 1 means no difference.

In [ ]:
from IPython.display import Image, display
display(Image(filename='plots/step5_cox_forest_plot.png'))

### What the model says

**Business type dominates.** Food & Beverage: HR = 1.96 — nearly double the closure risk of the baseline (Arts & Culture). Technology (1.88) and Retail (1.67) follow. Construction & Trades (HR = 0.55) is the most protective, though estimated on a non-representative subsample.

**Neighbourhood effects are real but smaller.** Most neighbourhoods are *safer* than Downtown — Marpole, Hastings-Sunrise, and Renfrew-Collingwood all at HR = 0.70 (30% lower risk). **Mount Pleasant is not significantly different from Downtown** (HR = 1.05, p = 0.10). That 7-year median from Step 3 was a composition effect.

**Land value adds a small signal.** HR = 1.11 per log-unit increase — about 7% more closure risk per doubling. That's the *opposite* direction from the raw correlation in Step 4. The Cox model unmasks the confounding: expensive areas have more durable business types. Once type is held constant, higher land value is a modest net negative. But this is a residential proxy, so the interpretation is suggestive.

### Hazard Ratios — Full Model

| Variable | HR | 95% CI | p-value | Direction |
|---|---|---|---|---|
| **Food & Beverage** | 1.96 | [1.81, 2.12] | < 0.001 | Higher risk |
| **Technology & Information** | 1.88 | [1.70, 2.07] | < 0.001 | Higher risk |
| **Retail** | 1.67 | [1.54, 1.82] | < 0.001 | Higher risk |
| **Professional Services** | 1.56 | [1.44, 1.69] | < 0.001 | Higher risk |
| **Manufacturing & Wholesale** | 1.50 | [1.36, 1.65] | < 0.001 | Higher risk |
| **Health & Beauty** | 1.29 | [1.19, 1.40] | < 0.001 | Higher risk |
| **Log Land Value** | 1.11 | [1.04, 1.18] | 0.002 | Higher risk |
| Financial & Insurance | 1.10 | [1.00, 1.21] | 0.058 | Not significant |
| Mount Pleasant | 1.05 | [0.99, 1.11] | 0.104 | Not significant |
| **Number of Employees** | 0.99 | [0.99, 0.99] | < 0.001 | Protective |
| Entry Year | 0.97 | [0.96, 0.97] | < 0.001 | See note |
| **West End** | 0.91 | [0.86, 0.96] | < 0.001 | Protective |
| **Riley Park** | 0.88 | [0.80, 0.97] | 0.007 | Protective |
| **Strathcona** | 0.87 | [0.81, 0.92] | < 0.001 | Protective |
| **Sunset** | 0.79 | [0.72, 0.87] | < 0.001 | Protective |
| **Fairview** | 0.79 | [0.75, 0.82] | < 0.001 | Protective |
| **Renfrew-Collingwood** | 0.70 | [0.64, 0.77] | < 0.001 | Protective |
| **Hastings-Sunrise** | 0.70 | [0.63, 0.77] | < 0.001 | Protective |
| **Marpole** | 0.70 | [0.62, 0.78] | < 0.001 | Protective |
| **Construction & Trades** | 0.55 | [0.50, 0.61] | < 0.001 | Protective* |

*Reference: Arts & Culture (type), Downtown (neighbourhood). Bold = significant at p < 0.05.*
*\*Estimated on 46% of category (commercial-address businesses only).*

In [ ]:
display(Image(filename='plots/step5_model_comparison.png'))

### How good is the model?

Concordance = 0.62 — better than chance but leaves most of the variance unexplained. Business failure depends on things this data can't see: management quality, capital reserves, lease terms, competition on the specific block.

**Known limitations:**

- **Left-truncation**: 40% of businesses were already operating in 2013. Their true tenure is unknown. Restricted cohort gives similar estimates.
- **Proportional hazards violations**: The hazard ratios aren't constant over time — the type effect likely shifts as businesses age. These are time-averaged effects.
- **Differential missingness**: 54% of Construction & Trades lacks a commercial address and is excluded from the Cox sample. The HR = 0.55 is estimated on the commercial-address half only.
- **Employee count**: 19% report zero employees — possibly sole proprietorships or missing data.
- **Entry year**: Later cohorts appear "protected" partly because their failures haven't happened yet within the observation window.

### Back to the ice cream shop

My friend's biggest risk factor is being Food & Beverage (HR = 1.96). His neighbourhood isn't the problem — Mount Pleasant is no more dangerous than Downtown once you control for business mix. The 6-year median stands. If he makes it past year 5, he's in the more durable half of his cohort. That's what I can tell him — with the honest caveat that the model explains 62% of the discrimination, and the rest is invisible.

---

*Data: City of Vancouver Open Data — Business Licences (2013–2026) + Property Tax Assessments (2006–2025). 97,915 independent businesses, 53,805 with complete data for regression. Code and data preparation on [GitHub](https://github.com/7AVS/portfolio).*